#Indexing: Inspecting and Managing Documents in Vectorstore

In [ ]:
%load_ext dotenv
%doenv

In [ ]:
from langchain.openai.embeddings import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.document_loaders import TextLoader
from langchain_core.documents import Document

In [ ]:
embedding = OpenAIEmbeddings(model = "text-embedding-ada-002")

In [ ]:
vectorstore_from_directory = Chroma(persist_directory = "./file name",
                                    embedding_function = embedding)

In [ ]:
vectorstore_from_directory.get()

In [ ]:
added_document = Document(page_content= "",
                          metadata = {'Topic': 'CONSCIOUSNESS AND THE NATURE OF REALITY',
                                      'subTopic': 'Artificial Intelligence and Consciousness'})

In [ ]:
vectorstore_from_directory.add_documents([added_document])

In [ ]:
question = "What is the difference between artificial intelligence and consciousness?"

In [ ]:
retrived_docs = vectorstore_from_directory.similarity_search(query = question,
                                                             k = 5)

In [ ]:
retrived_docs

In [ ]:
for i in retrived_docs:
  print(f"Page content:{i.page_content}\n------------\n")

In [ ]:
retrived_docs = vectorstore_from_directory.max_marginal_relavence_search(query = question,
                                                                         k = 5,
                                                                         lambda_mult = 0.5,
                                                                         filter = {'subTopic':''})

In [ ]:
for i in retrived_docs:
  print(f"Page content:{i.page_content}\n------------\n")

#RAG retrive

In [ ]:
retriver = vectorestore_from_directory.as_retriever(search_type = 'mmr',
                                                    search_kwargs = {'k':3,
                                                                     'lambda_mult':0.7})

In [ ]:
question = "What is quantam ... ?"
retrived_docs = retriever.invoke(question)

In [ ]:
for i in retrived_docs:
  print(f"Page content:{i.page_content}\n------------\n")

#RAG PIPELINE

In [ ]:
TEMPLATE = '''
Answer the following question.
{question}

To answer the questions, use only following context:
{context}

At the end of the response, specify the name of the lecture this context is taken from in the format:
Resources:*SubTopic*
Where *SupTopic* should be substituted with the SubTopic of all resesources'''
prompt_template = PromptTemplate.from_template(TEMPLATE)

In [ ]:
chat = ChatOpenAI(model = '',
                  seed = 42,
                  max_token = 200)

In [ ]:
question = ""

In [ ]:
chain = ({'context': retriver,
          'question': RunnablePassthrough()} | prompt_template | chat)

In [ ]:
chain.invoke(question)